# OSCD CPU benchmark
Run all cells on a fresh Colab runtime. Threshold choices use only training cities; the official test split is scored once at the end.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/oscd'
REPO_URL = 'https://github.com/Aravind-Subramanian-16/satellite-change-detection.git'
REPO_DIR = '/content/repo'
OSCD_DIR = '/content/oscd'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
drive_dir = Path(DRIVE_DIR)
for weight in ('model.onnx', 'best_model.pt'):
    source = drive_dir / 'weights' / weight
    if not source.is_file():
        raise FileNotFoundError(f'Missing required weight: {source}')

!mkdir -p /content/oscd
!unzip -q -o "$DRIVE_DIR/archive (1).zip" -d /content/oscd
!unzip -q -o "$DRIVE_DIR/test_labels.zip" -d /content/oscd

if Path(REPO_DIR).is_dir():
    %cd /content/repo
    !git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/repo

!cp "$DRIVE_DIR/weights/model.onnx" /content/repo/model.onnx
!cp "$DRIVE_DIR/weights/best_model.pt" /content/repo/best_model.pt
!grep -viE "^(onnx==|numpy)" requirements.txt > /content/req_colab.txt
!python -m pip -q install -r /content/req_colab.txt
!python -m pytest -q


In [ ]:
from pathlib import Path
images = Path(OSCD_DIR) / 'images' / 'Onera Satellite Change Detection dataset - Images'
train = [city.strip() for city in (images / 'train.txt').read_text().replace('\n', ',').split(',') if city.strip()]
test = [city.strip() for city in (images / 'test.txt').read_text().replace('\n', ',').split(',') if city.strip()]
print(f'Train cities ({len(train)}): {train}')
print(f'Test cities ({len(test)}): {test}')
assert len(train) == 14 and len(test) == 10
for split, cities, labels in [('train', train, Path(OSCD_DIR) / 'train_labels' / 'Onera Satellite Change Detection dataset - Train Labels'), ('test', test, Path(OSCD_DIR) / 'Onera Satellite Change Detection dataset - Test Labels')]:
    for city in cities:
        for directory in (images / city / 'imgs_1_rect', images / city / 'imgs_2_rect'):
            assert directory.is_dir(), f'{split} {city}: missing bands at {directory}'
        assert (labels / city / 'cm' / 'cm.png').is_file(), f'{split} {city}: missing label'
print('All cities have both acquisitions and a label.')


In [ ]:
%cd /content/repo
# Results are checkpointed under Drive after every city; rerun resumes completed rows.
!python benchmarks/oscd_eval.py --root /content/oscd --output "$DRIVE_DIR/results"
!cat "$DRIVE_DIR/results/oscd_results.md"
